In [ ]:
from pathlib import Path
from loguru import logger
import sys
sys.path.append("../..")

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader

from auto_encoder.dataset import BBoxDataset
from auto_encoder.loss_utils import ae_loss, vae_loss
from auto_encoder.models import Autoencoder_v1, Autoencoder_v2, Autoencoder_VAE_v1
from auto_encoder.utils import show_reconstruction_from_dataset
from auto_encoder.train import train_autoencoder

from constants import IMAGE_DIR_PATH

In [ ]:
annotations_path=annotations_path=Path("labels/auto_annotations_fire_watcher-1.json")

dataset_raw = BBoxDataset(image_dir=IMAGE_DIR_PATH,annotations_path=annotations_path,)

# Autoencoder v1

In [ ]:
model = Autoencoder_v1(latent_dim=64)
model_path = Path("models/autoencoder_2k.pth")
state_dict = torch.load(model_path, map_location="cpu")
model.load_state_dict(state_dict)
# model.eval()

In [ ]:
for i in [
    # "screenshot_2026-01-02_10-20-01.jpg", # day, no fire
    # "screenshot_2026-01-01_15-30-00.jpg", # evening, small fire big log
    # "screenshot_2026-01-02_17-35-00.jpg", # night, big fire
    # "screenshot_2026-01-10_21-55-01.jpg", # night greenish, small fire
    # # unseen images
    # "screenshot_2026-01-18_22-34-31.jpg", 
    # "screenshot_2026-01-18_22-41-01.jpg", # person

    ]:
    print(i)
    show_reconstruction_from_dataset(
        model,
        dataset_raw,
        i,
        device="cpu",
    )


# Train VAE

In [ ]:
model, dataset = train_autoencoder(
    image_dir=IMAGE_DIR_PATH,
    annotations_path=Path("labels/auto_annotations_fire_watcher-1.json"),
    model_cls=Autoencoder_VAE_v1,
    loss_fn=vae_loss,
    max_images=2000,
    epochs=30,
    batch_size=8,
    latent_dim=16,
    device="cpu",
    lr = 1e-3,
)

torch.save(model.state_dict(), "models/autoencoder_vae_v2.pth")
